In [1]:
import json
from collections import Counter

In [2]:
with open("baseline_results.json", "r", encoding="utf-8") as f:
    baseline_results = json.load(f)

with open("retry_results.json", "r", encoding="utf-8") as f:
    retry_results = json.load(f)

In [3]:
baseline_total = len(baseline_results)
baseline_valid = sum(1 for r in baseline_results if r["json_valid"])
baseline_json_valid_rate = baseline_valid / baseline_total

baseline_non_answerable = [
    r for r in baseline_results
    if (r["answerable"] is False and r["json_valid"] and r["parsed"] is not None)
]

baseline_refusal_correct = sum(
    1 for r in baseline_non_answerable
    if r["parsed"]["refused"] is True
)

baseline_refusal_accuracy = (
    baseline_refusal_correct / len(baseline_non_answerable)
    if baseline_non_answerable else None
)

baseline_hallucinations = sum(
    1 for r in baseline_non_answerable
    if r["parsed"]["refused"] is False
)

baseline_hallucination_rate = (
    baseline_hallucinations / len(baseline_non_answerable)
    if baseline_non_answerable else None
)

print("Baseline JSON validity rate:", round(baseline_json_valid_rate, 3))
print("Baseline refusal accuracy:", None if baseline_refusal_accuracy is None else round(baseline_refusal_accuracy, 3))
print("Baseline hallucination rate:", None if baseline_hallucination_rate is None else round(baseline_hallucination_rate, 3))

Baseline JSON validity rate: 0.9
Baseline refusal accuracy: 0.25
Baseline hallucination rate: 0.75


In [4]:
def extract_first_json_block(text: str):
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return None


retry_total = len(retry_results)
retry_valid = sum(1 for r in retry_results if r["valid"])
retry_json_valid_rate = retry_valid / retry_total

retry_non_answerable = []
for r in retry_results:
    if r["answerable"] is False and r["valid"]:
        block = extract_first_json_block(r["final_output"])
        if block:
            try:
                parsed = json.loads(block)
                retry_non_answerable.append({
                    **r,
                    "parsed": parsed
                })
            except:
                pass

retry_refusal_correct = sum(
    1 for r in retry_non_answerable
    if r["parsed"]["refused"] is True
)

retry_refusal_accuracy = (
    retry_refusal_correct / len(retry_non_answerable)
    if retry_non_answerable else None
)

retry_hallucinations = sum(
    1 for r in retry_non_answerable
    if r["parsed"]["refused"] is False
)

retry_hallucination_rate = (
    retry_hallucinations / len(retry_non_answerable)
    if retry_non_answerable else None
)

print("Retry JSON validity rate:", round(retry_json_valid_rate, 3))
print("Retry refusal accuracy:", None if retry_refusal_accuracy is None else round(retry_refusal_accuracy, 3))
print("Retry hallucination rate:", None if retry_hallucination_rate is None else round(retry_hallucination_rate, 3))

Retry JSON validity rate: 0.0
Retry refusal accuracy: None
Retry hallucination rate: None


In [5]:
total_retries = sum(r["retries"] for r in retry_results)
avg_retries = total_retries / len(retry_results)

retry_distribution = Counter(r["retries"] for r in retry_results)

print("Total retries used:", total_retries)
print("Average retries per question:", round(avg_retries, 3))
print("Retry distribution:", dict(retry_distribution))

Total retries used: 40
Average retries per question: 2.0
Retry distribution: {2: 20}


In [6]:
summary = {
    "baseline_json_validity_rate": round(baseline_json_valid_rate, 3),
    "retry_json_validity_rate": round(retry_json_valid_rate, 3),
    "baseline_refusal_accuracy": None if baseline_refusal_accuracy is None else round(baseline_refusal_accuracy, 3),
    "retry_refusal_accuracy": None if retry_refusal_accuracy is None else round(retry_refusal_accuracy, 3),
    "baseline_hallucination_rate": None if baseline_hallucination_rate is None else round(baseline_hallucination_rate, 3),
    "retry_hallucination_rate": None if retry_hallucination_rate is None else round(retry_hallucination_rate, 3),
    "total_retries": total_retries,
    "average_retries": round(avg_retries, 3)
}

summary

{'baseline_json_validity_rate': 0.9,
 'retry_json_validity_rate': 0.0,
 'baseline_refusal_accuracy': 0.25,
 'retry_refusal_accuracy': None,
 'baseline_hallucination_rate': 0.75,
 'retry_hallucination_rate': None,
 'total_retries': 40,
 'average_retries': 2.0}